In [ ]:
# Celda 1: Entorno y carga
import runpy
from pathlib import Path
from pyspark.sql import functions as F

runpy.run_path(str(next(p / "notebooks" / "_init_env.py" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "notebooks" / "_init_env.py").is_file())))

from src.catalog import SILVER_AEMET_TABLE, SILVER_FIRMS_TABLE

df_firms_day = (
    spark.table(SILVER_FIRMS_TABLE)
    .groupBy("acq_date")
    .agg(
        F.count("*").alias("detecciones_firms"),
        F.round(F.sum("frp"), 2).alias("frp_total"),
    )
)

df_aemet_day = (
    spark.table(SILVER_AEMET_TABLE)
    .groupBy("fecha_dt")
    .agg(
        F.round(F.max("dias_secos_consecutivos"), 1).alias("max_dias_secos"),
        F.round(F.avg("tmax_c"), 1).alias("tmax_media_c"),
        F.round(F.min("hr_min_pct"), 1).alias("hr_min_extrema"),
    )
)

In [ ]:
# Celda 2: Días con mayor actividad térmica y su contexto meteorológico
df_joined = (
    df_firms_day.join(df_aemet_day, df_firms_day["acq_date"] == df_aemet_day["fecha_dt"], "left")
    .drop("fecha_dt")
)

display(df_joined.orderBy(F.desc("frp_total")).limit(15))

In [ ]:
# Celda 3: Correlación simple entre rachas secas y actividad térmica
display(
    df_joined.filter(F.col("max_dias_secos").isNotNull())
    .withColumn("tramo_sequia", F.when(F.col("max_dias_secos") >= 15, ">= 15 días").otherwise("< 15 días"))
    .groupBy("tramo_sequia")
    .agg(
        F.count("*").alias("dias_con_detecciones"),
        F.round(F.avg("detecciones_firms"), 1).alias("media_detecciones"),
        F.round(F.avg("frp_total"), 1).alias("media_frp_total"),
    )
)